## Data Availability Notice

The data used in this notebook was provided by **Dogs Trust** under a formal data sharing agreement as part of an MSc Data Science dissertation at Liverpool John Moores University, supervised by Dr Jon Bielby.

**The raw data files are not included in this repository.** To reproduce the analysis, the following data extracts from the Dogs Trust Salesforce CRM are required:

- `Dog Affiliation and Status_Sub Status History Pull.xlsx` (Affiliation History sheet)
- `Sub Status History v2.csv`

Requests for data access should be directed to the Dogs Trust Data Science & Analytics team.

All data handling was conducted in accordance with GDPR and Dogs Trust's data governance protocols.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Load data
affiliation_history = pd.read_excel('../data/[Dogs Trust data file - not included in repo]', sheet_name='Affiliation History')

print(affiliation_history.shape)

In [ ]:
# Filter to Intake records only
intake_records = affiliation_history[affiliation_history['Process'] == 'Intake'].copy()

# Convert Start Date to datetime
intake_records['Start Date'] = pd.to_datetime(intake_records['Start Date'])

# Count intakes per month
intake_records['Year Month'] = intake_records['Start Date'].dt.to_period('M')
monthly_arrivals = intake_records.groupby('Year Month').size().reset_index(name='Intake Count')

print(monthly_arrivals)
print(f"\nAverage monthly arrivals: {monthly_arrivals['Intake Count'].mean():.1f}")
print(f"Std deviation: {monthly_arrivals['Intake Count'].std():.1f}")
print(f"Min: {monthly_arrivals['Intake Count'].min()}, Max: {monthly_arrivals['Intake Count'].max()}")

In [ ]:
# Filter to complete months only (exclude 2026 as incomplete)
complete_months = monthly_arrivals[monthly_arrivals['Year Month'] < pd.Period('2026-01', 'M')]

print(f"Complete months: {len(complete_months)}")
print(f"\nAverage monthly arrivals: {complete_months['Intake Count'].mean():.1f}")
print(f"Std deviation: {complete_months['Intake Count'].std():.1f}")
print(f"Min: {complete_months['Intake Count'].min()}, Max: {complete_months['Intake Count'].max()}")
print(f"\nAverage weekly arrivals: {complete_months['Intake Count'].mean() / 4.33:.1f}")

In [ ]:
fig_arrivals = px.line(
    complete_months,
    x=complete_months['Year Month'].astype(str),
    y='Intake Count',
    title='Monthly Dog Intake - Dogs Trust (Jan 2024 - Dec 2025)',
    labels={'x': 'Month', 'Intake Count': 'Number of Intakes'},
    markers=True
)

fig_arrivals.add_hline(
    y=complete_months['Intake Count'].mean(),
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean: {complete_months['Intake Count'].mean():.0f}"
)

fig_arrivals.update_layout(
    height=450,
    width=900)
fig_arrivals.show()

In [ ]:
fig_arrivals.write_image('../outputs/monthly_dog_intake.png', width=1000, height=600, scale=2)
fig_arrivals.write_html('../outputs/monthly_dog_intake.html') # Interactive version

In [ ]:
# Sort by dog and date
sorted_aff = affiliation_history.sort_values(by=['Dog ID', 'Start Date'])

# Get next process for each record
sorted_aff['Next Process'] = sorted_aff.groupby('Dog ID')['Process'].shift(-1)

# Drop last record for each dog (no next process)
transitions = sorted_aff.dropna(subset=['Next Process'])

# Filter to known processes only
known_processes = ['Intake', 'Movement', 'Transfer', 'Adoption', 'Return']
transitions = transitions[
    transitions['Process'].isin(known_processes) & 
    transitions['Next Process'].isin(known_processes)
]

# Calculate transition probabilities
transition_prob = transitions.groupby(['Process', 'Next Process']).size().reset_index(name='Count')
transition_prob['Probability'] = transition_prob.groupby('Process')['Count'].transform(lambda x: x / x.sum())
transition_prob['Probability'] = transition_prob['Probability'].round(4)

print(transition_prob.sort_values(['Process', 'Probability'], ascending=[True, False]))

In [ ]:
import plotly.express as px
import pandas as pd

# Create pivot table for heatmap
heatmap_data = transition_prob.pivot(index='Process', columns='Next Process', values='Probability').fillna(0)

# Reorder rows and columns in journey order
order = ['Intake', 'Movement', 'Transfer', 'Adoption', 'Return']
heatmap_data = heatmap_data.reindex(index=order, columns=order)

fig_heatmap = px.imshow(
    heatmap_data,
    text_auto=' .3f',
    color_continuous_scale='Reds',
    labels=dict(x='Next Process', y='Current Process', color='Probability'),
    title="Transition Probabilities Matrix - Dog Journey Processes"
)

fig_heatmap.update_layout(
    height=450,
    width=700,
    xaxis_title='Next Process',
    yaxis_title='Current Process'
)

fig_heatmap.show()

In [ ]:
fig_heatmap.write_image('../outputs/transition_probabilities_heatmap.png', width=900, height=550, scale=2)
fig_heatmap.write_html('../outputs/transition_probabilities_heatmap.html') # Interactive version

In [ ]:

# Calculate durations of each stage in days
sorted_aff['Start Date'] = pd.to_datetime(sorted_aff['Start Date'])
sorted_aff['End Date'] = pd.to_datetime(sorted_aff['End Date'])

# Duration = End Date - Start Date
sorted_aff['Duration Days'] = (
    sorted_aff['End Date'] - sorted_aff['Start Date']
).dt.days

# Drop records with no End Date or negative duration
stage_durations = sorted_aff[
    sorted_aff['Duration Days'].notna() &
    sorted_aff['Duration Days'] >= 0
].copy()

# Summary stats per process
duration_stats = stage_durations.groupby('Process')['Duration Days'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
    ).round(1)

print(duration_stats)

In [ ]:
from scipy import stats
import numpy as np

for process in ['Intake', 'Movement']:
    data = stage_durations[stage_durations['Process'] == process]['Duration Days'].dropna()
    data = data[data > 0]  # Exclude zero durations for lognormal fitting

    # Fit exponential, gamma and lognormal
    exp_params = stats.expon.fit(data, floc=0)
    gamma_params = stats.gamma.fit(data, floc=0)
    lognorm_params = stats.lognorm.fit(data, floc=0)

    # KS test for goodness of fit
    exp_ks = stats.kstest(data, 'expon', args=exp_params)
    gamma_ks = stats.kstest(data, 'gamma', args=gamma_params)
    lognorm_ks = stats.kstest(data, 'lognorm', args=lognorm_params)

    print(f"\n{process} Duration Distribution Fit:")
    print(f" Exponential - KS Statistic: {exp_ks.statistic:.4f}, p-value: {exp_ks.pvalue:.4f}")
    print(f" Gamma - KS Statistic: {gamma_ks.statistic:.4f}, p-value: {gamma_ks.pvalue:.4f}")
    print(f" Lognormal - KS Statistic: {lognorm_ks.statistic:.4f}, p-value: {lognorm_ks.pvalue:.4f}")
    print(f" Best fit: {min(['Exponential', 'Gamma', 'Log-normal'],
                             key=lambda x: [exp_ks.statistic, gamma_ks.statistic, lognorm_ks.statistic]
                             [['Exponential', 'Gamma', 'Log-normal'].index(x)])}")

In [ ]:
from scipy import stats

# Intake - Gamma
intake_data = stage_durations[stage_durations['Process'] == 'Intake']['Duration Days'].dropna()
intake_data = intake_data[intake_data > 0]
gamma_params = stats.gamma.fit(intake_data, floc=0)
print(f"Intake Gamma params - shape (a): {gamma_params[0]:.4f}, scale: {gamma_params[2]:.4f}")

# Movement - Lognormal
movement_data = stage_durations[stage_durations['Process'] == 'Movement']['Duration Days'].dropna()
movement_data = movement_data[movement_data > 0]
lognorm_params = stats.lognorm.fit(movement_data, floc=0)
print(f"Movement Lognormal params - shape (s): {lognorm_params[0]:.4f}, scale: {lognorm_params[2]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Intake - Gamma fit
intake_data_plot = intake_data[intake_data <= 200] # trim extreme outliers
x = np.linspace(0, 200, 200)
axes[0].hist(intake_data_plot, bins=50, density=True, alpha=0.6, color='steelblue', label='Actual data')
axes[0].plot(x, stats.gamma.pdf(x, a=0.9319, scale=26.4569), 'r-', lw=2, label='Gamma fit')
axes[0].set_title('Intake Duration - Gamma Fit')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Density')
axes[0].legend()

# Movement - Lognormal fit
movement_data_plot = movement_data[movement_data <=200]
axes[1].hist(movement_data_plot, bins=50, density=True, alpha=0.6, color='salmon', label='Actual data')
axes[1].plot(x, stats.lognorm.pdf(x, s=1.0914, scale=11.2346), 'b-', lw=2, label='Log-normal fit')
axes[1].set_title('Movement Durations - Log-normal Fit')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/distribution_fits.png', dpi=150, bbox_inches='tight')
plt.show()